**Como executar o teste**

Usamos este notebook para comparar as tabelas antes e depois de rodar o pipeline novamente com os mesmos arquivos. A ação é definida pela variável `acao`, na primeira célula de código.

1. Execute o Job completo e aguarde Bronze, Silver e Gold terminarem com sucesso.
2. Na primeira célula deste notebook, defina `acao = "registrar_referencia"` e execute todas as células. Isso salva as versões das tabelas e os hashes dos arquivos que serão usados na comparação.
3. Execute novamente o mesmo Job, mantendo os arquivos de entrada e os códigos.
4. Quando o Job terminar, volte a este notebook, altere a variável para `acao = "comparar"` e execute todas as células.
5. Confira o relatório. O esperado é que as 22 tabelas tenham `aprovado = true`, com as mesmas contagens e nenhuma linha ausente ou adicional.

Deixamos `comparar` como valor padrão. A opção `registrar_referencia` deve ser usada no início de um novo teste, pois substitui a referência salva anteriormente.

Este notebook é executado separadamente do Job recorrente.

**Premissa de auditoria**

A coluna `_ingerido_em` registra o horário da carga na Bronze e é preservada na Silver. Esse horário é atualizado quando rodamos a ingestão novamente, mesmo com os mesmos arquivos.

Na Gold, a coluna `_gold_atualizado_em` registra o horário de atualização e também muda a cada execução.

Na comparação, desconsideramos somente os valores de `_ingerido_em` na Bronze/Silver e de `_gold_atualizado_em` na Gold. A presença e o tipo dessas colunas continuam sendo conferidos. Todas as outras colunas são comparadas normalmente, incluindo a quantidade de ocorrências de cada linha.

O objetivo é verificar se a reexecução mantém os dados de negócio, permitindo a atualização esperada do horário de auditoria.

In [0]:
from pathlib import Path
import hashlib
from pyspark.sql import functions as sf

acao = "comparar"

catalogo = "vcredit_medallion"
schema_auditoria = f"{catalogo}.auditoria"
landing_path = f"/Volumes/{catalogo}/bronze/landing"
tabela_versoes = f"{schema_auditoria}.idem_versoes_referencia"
tabela_arquivos = f"{schema_auditoria}.idem_arquivos_referencia"
tabela_resultados = f"{schema_auditoria}.idem_resultados"

bases = ["base_atendentes", "base_motivos", "canais", "chamados", "chamados_hora",
         "clientes", "custos", "pesquisa_satisfacao"]
tabelas_delta = [f"{catalogo}.{camada}.tb_{base}" for camada in ["bronze", "silver"]
                for base in bases]
tabelas_delta += [f"{catalogo}.gold.{nome}" for nome in [
    "dim_motivo", "dim_canal", "dim_atendente", "dim_cliente", "dim_data", "fato_chamados"]]

def exigir(condicao, descricao):
    if not condicao:
        raise ValueError(descricao)
    print(f"OK: {descricao}")

def versao_atual(tabela):
    return int(spark.sql(f"DESCRIBE HISTORY {tabela} LIMIT 1").first()["version"])

def hashes_arquivos():
    resultados = []
    for base in bases:
        arquivo = f"{base}.csv"
        caminho = Path(landing_path) / arquivo
        digest = hashlib.sha256()
        with caminho.open("rb") as entrada:
            for bloco in iter(lambda: entrada.read(1024 * 1024), b""):
                digest.update(bloco)
        resultados.append((arquivo, digest.hexdigest()))
    return resultados

def ordenar_colunas(df):
    return df.select(*sorted(df.columns))

def colunas_auditoria_ignoradas(tabela):
    camada = tabela.split(".")[1]

    if camada in ["bronze", "silver"]:
        return ["_ingerido_em"]

    if camada == "gold":
        return ["_gold_atualizado_em"]

    return []

def comparar_tabela(tabela, v_antes, v_depois):
    antes = ordenar_colunas(spark.read.option("versionAsOf", v_antes).table(tabela))
    depois = ordenar_colunas(spark.read.option("versionAsOf", v_depois).table(tabela))
    ignoradas = colunas_auditoria_ignoradas(tabela)
    auditoria_presente = all(c in antes.columns and c in depois.columns for c in ignoradas)
    schema_igual = (antes.schema.simpleString() == depois.schema.simpleString()
                    and auditoria_presente)
    n_antes, n_depois = antes.count(), depois.count()
    if schema_igual:
        colunas_negocio = [c for c in antes.columns if c not in ignoradas]
        negocio_antes = antes.select(*colunas_negocio)
        negocio_depois = depois.select(*colunas_negocio)
        ausentes = negocio_antes.exceptAll(negocio_depois).count()
        adicionais = negocio_depois.exceptAll(negocio_antes).count()
    else:
        ausentes, adicionais = -1, -1
    return (tabela, v_antes, v_depois, n_antes, n_depois,
            schema_igual, ausentes, adicionais, ", ".join(ignoradas), v_depois > v_antes)

exigir(acao in ["registrar_referencia", "comparar"], "Ação reconhecida.")
exigir(len(tabelas_delta) == 22 and len(set(tabelas_delta)) == 22,
       "Escopo definido: 22 tabelas distintas de Bronze, Silver e Gold.")
tabelas_ausentes = [t for t in tabelas_delta if not spark.catalog.tableExists(t)]
exigir(not tabelas_ausentes, f"Todas as tabelas do pipeline existem. Ausentes: {tabelas_ausentes}.")


OK: Ação reconhecida.
OK: Escopo definido: 22 tabelas distintas de Bronze, Silver e Gold.
OK: Todas as tabelas do pipeline existem. Ausentes: [].


## 2. Registrar ou comparar

**Registrar:** guardar as versões das 22 tabelas e os hashes dos oito arquivos em duas tabelas de controle. Confirmamos que as fontes não mudaram durante o registro. A referência usa histórico Delta, sem duplicar os registros das fontes.

**Comparar:** validar os hashes de entrada; ler cada tabela na versão de referência e na versão atual; conferir o schema completo e comparar o conteúdo com `exceptAll` nos dois sentidos, retirando somente os valores do metadado explicado acima. A comparação considera a multiplicidade das linhas: duplicatas extras são detectadas mesmo quando se usa o mesmo identificador. Igualdade de contagem ou soma, sozinha, seria insuficiente. Igualdade de todo o conteúdo de custos também preserva os seus totais.

Com os notebooks atuais, cada tabela é gravada por `overwrite` em cada execução. Exigimos que a versão de cada uma tenha avançado para evitar aprovar uma comparação feita sem reexecutar o fluxo. A versão pode aumentar sem alterar o conteúdo: o critério é nova gravação com schema e conteúdo de negócio iguais. O avanço de versão não substitui os dois registros de execução bem-sucedida do Job. Se o schema mudar, o relatório marca falha e usa `-1` nas diferenças, pois o conteúdo não foi comparado.

Não rode cargas ou alterações nas tabelas durante o registro ou a comparação. O teste interrompe se detectar mudanças concorrentes. Se futuramente o pipeline passar a evitar gravações quando a entrada não mudar, a exigência de avanço de versão deverá ser revista.


In [0]:
print("Ação em execução:", acao)

if acao == "registrar_referencia":
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema_auditoria}")
    arquivos_inicio = hashes_arquivos()
    versoes = [(t, versao_atual(t)) for t in tabelas_delta]

    exigir(all(versao_atual(t) == v for t,v in versoes),
           "As tabelas não mudaram durante o registro da referência.")
    exigir(arquivos_inicio == hashes_arquivos(),
           "Os arquivos não mudaram durante o registro da referência.")
    (spark.createDataFrame(versoes, "tabela string, versao_referencia long")
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(tabela_versoes))
    (spark.createDataFrame(arquivos_inicio, "arquivo string, sha256 string")
        .write.format("delta").mode("overwrite").option("overwriteSchema", "true")
        .saveAsTable(tabela_arquivos))
    display(spark.table(tabela_versoes).orderBy("tabela"))
    print("Referência registrada. Execute o Job completo novamente; depois altere acao para 'comparar' na primeira célula e execute todas as células.")

else:
    exigir(spark.catalog.tableExists(tabela_versoes)
        and spark.catalog.tableExists(tabela_arquivos),
        "Referência existente. Se não existir, registre após a primeira execução do Job.")
    referencia = spark.table(tabela_versoes).collect()
    exigir(len(referencia) == len(tabelas_delta)
        and {r["tabela"] for r in referencia} == set(tabelas_delta),
        "O conjunto de tabelas de referência corresponde ao conjunto testado.")
    hashes_antes = {r["arquivo"]:r["sha256"] for r in spark.table(tabela_arquivos).collect()}
    hashes_depois = dict(hashes_arquivos())
    exigir(hashes_antes == hashes_depois,
        "Os oito CSVs são idênticos à referência. Se mudaram, este não é o mesmo teste de entrada.")

    resultados = []
    versoes_comparacao = [(r["tabela"], versao_atual(r["tabela"])) for r in referencia]
    versoes_atuais = dict(versoes_comparacao)
    for r in referencia:
        tabela = r["tabela"]
        v_antes = int(r["versao_referencia"])
        v_depois = versoes_atuais[tabela]
        resultados.append(comparar_tabela(tabela, v_antes, v_depois))

    exigir(all(versao_atual(t) == v for t,v in versoes_comparacao),
           "As fontes permaneceram estáveis durante a comparação.")
    exigir(hashes_depois == dict(hashes_arquivos()),
           "Os arquivos permaneceram estáveis durante a comparação.")
    df_resultados = spark.createDataFrame(resultados, """objeto string,
        versao_antes long, versao_depois long, registros_antes long,
        registros_depois long, schema_igual boolean, linhas_ausentes long,
        linhas_adicionais long, colunas_auditoria_ignoradas string, versao_avancou boolean""")
    df_resultados = df_resultados.withColumn("aprovado",
        sf.col("schema_igual") & (sf.col("registros_antes") == sf.col("registros_depois"))
        & (sf.col("linhas_ausentes") == 0) & (sf.col("linhas_adicionais") == 0)
        & sf.col("versao_avancou"))
    (df_resultados.write.format("delta").mode("overwrite")
        .option("overwriteSchema", "true").saveAsTable(tabela_resultados))
    display(df_resultados.orderBy("objeto"))
    falhas = df_resultados.filter(~sf.col("aprovado")).count()
    exigir(falhas == 0, f"Idempotência dos dados de negócio nas 22 tabelas. Falhas: {falhas}. "
           "Se versao_avancou=false, confirme que o Job completo foi reexecutado após o registro.")
    print("Teste concluído: 22 tabelas aprovadas. "
    "Foram desconsiderados somente os valores de _ingerido_em na Bronze/Silver "
    "e de _gold_atualizado_em na Gold. "
    "O schema completo e os dados de negócio foram verificados.")


Ação em execução: comparar
OK: Referência existente. Se não existir, registre após a primeira execução do Job.
OK: O conjunto de tabelas de referência corresponde ao conjunto testado.
OK: Os oito CSVs são idênticos à referência. Se mudaram, este não é o mesmo teste de entrada.
OK: As fontes permaneceram estáveis durante a comparação.
OK: Os arquivos permaneceram estáveis durante a comparação.


objeto,versao_antes,versao_depois,registros_antes,registros_depois,schema_igual,linhas_ausentes,linhas_adicionais,colunas_auditoria_ignoradas,versao_avancou,aprovado
vcredit_medallion.bronze.tb_base_atendentes,11,12,25,25,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_base_motivos,11,12,14,14,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_canais,11,12,4,4,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_chamados,13,14,134474,134474,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_chamados_hora,11,12,131838,131838,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_clientes,11,12,47705,47705,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_custos,11,12,131069,131069,true,0,0,_ingerido_em,true,true
vcredit_medallion.bronze.tb_pesquisa_satisfacao,11,12,81021,81021,true,0,0,_ingerido_em,true,true
vcredit_medallion.gold.dim_atendente,8,9,26,26,true,0,0,_gold_atualizado_em,true,true
vcredit_medallion.gold.dim_canal,8,9,4,4,true,0,0,_gold_atualizado_em,true,true


OK: Idempotência dos dados de negócio nas 22 tabelas. Falhas: 0. Se versao_avancou=false, confirme que o Job completo foi reexecutado após o registro.
Teste concluído: 22 tabelas aprovadas. Foram desconsiderados somente os valores de _ingerido_em na Bronze/Silver e de _gold_atualizado_em na Gold. O schema completo e os dados de negócio foram verificados.


## 3. Como interpretar e entregar a evidência

Aprovação exige 22 linhas com `aprovado = true`, `schema_igual = true`, `versao_avancou = true`, contagens iguais e zero linhas ausentes/adicionais. `colunas_auditoria_ignoradas` mostra `_ingerido_em` na Bronze/Silver e `_gold_atualizado_em` na Gold.

Se houver diferenças de conteúdo, investigue as tabelas com `aprovado = false` antes de gerar uma nova referência. Não registre outra referência imediatamente após uma falha: isso apagaria a base da comparação que você precisa analisar.

Para consultar o relatório depois, execute `SELECT * FROM vcredit_medallion.auditoria.idem_resultados ORDER BY objeto`.

O teste demonstra reexecução sobre a entrada e o código mantidos no ensaio. Ele não garante atomicidade entre todas as tabelas nem que uma carga futura estará correta. Os controles de qualidade próprios de cada notebook continuam necessários.

Guarde a configuração do Job, o agendamento com fuso, as duas execuções bem-sucedidas e o relatório. Execute este teste manual separadamente: o agendamento normal deve aceitar novos dados quando os CSVs forem atualizados.

O teste foi executado no Databricks com os mesmos oito arquivos de entrada. As 22 tabelas foram aprovadas, sem diferenças nos dados de negócio, considerando a premissa de auditoria descrita neste notebook.
